# A statistical Part-of-Speech (POS) tagged parser - Penn Treebank

Name: K. V. D. Kekulawala 

Registration Number: MS26902784

### Environment setup

In [156]:
# !pip install nltk scikit-learn

In [157]:
import nltk

for pkg in ['treebank', 'punkt', 'punkt_tab', 'universal_tagset']:
    nltk.download(pkg, quiet=True)

# create the folders  
import os
for d in ['data', 'models', 'results']:
    os.makedirs(d, exist_ok=True)

print('Setup done. NLTK version:', nltk.__version__)

Setup done. NLTK version: 3.9.2


### Load and explore the Penn Treebank

In [158]:
from nltk.corpus import treebank

tagged_sents = treebank.tagged_sents()
parsed_sents = treebank.parsed_sents()

print('Number of sentences :', len(tagged_sents))
print('Number of tokens    :', sum(len(s) for s in tagged_sents))
print()
print('Example tagged sentence (first 12 tokens):')
print(tagged_sents[0][:12])

Number of sentences : 3914
Number of tokens    : 100676

Example tagged sentence (first 12 tokens):
[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT'), ('board', 'NN'), ('as', 'IN')]


In [159]:
# The same sentence as a parse tree. .pretty_print() draws it as ASCII art.
print('Example parse tree:\n')
parsed_sents[0].pretty_print()

Example parse tree:

                                                     S                                                                         
                         ____________________________|_______________________________________________________________________   
                        |                                               VP                                                   | 
                        |                        _______________________|___                                                 |  
                      NP-SBJ                    |                           VP                                               | 
         _______________|___________________    |     ______________________|______________________________________          |  
        |          |              ADJP      |   |    |        |                PP-CLR                              |         | 
        |          |           ____|____    |   |    |        |          _______

### Pre-processing the trees

In [160]:
from nltk import Tree

def simplify_label(label):
    "NP-SBJ -> NP, ADVP-TMP-1 -> ADVP. Keep atomic tags like -NONE-, -LRB-."
    if len(label) > 1 and label[0] == '-' and label[-1] == '-':
        return label
    for i, ch in enumerate(label):
        if i > 0 and ch in '-=':
            return label[:i]
    return label

def to_tag_tree(tree):
    "Return a copy of the tree whose leaves are POS tags; drop -NONE- traces."
    if isinstance(tree, str):
        return tree
    label = simplify_label(tree.label())
    if len(tree) == 1 and isinstance(tree[0], str):        # a (POS word) leaf
        if tree.label() == '-NONE-':
            return None
        return Tree(label, [label])                        # -> (POS 'POS')
    kids = [k for k in (to_tag_tree(c) for c in tree) if k is not None]
    return Tree(label, kids) if kids else None

def strip_traces(tagged_sent):
    "Remove -NONE- tokens from a list of (word, tag) pairs."
    return [(w, t) for (w, t) in tagged_sent if t != '-NONE-']

# demo: see the effect on the first sentence
print('Original preterminals :', [t.label() for t in parsed_sents[0].subtrees()
                                  if len(t) == 1 and isinstance(t[0], str)][:8])
print('Tag-tree (top rules)  :')
to_tag_tree(parsed_sents[0]).pretty_print()

Original preterminals : ['NNP', 'NNP', ',', 'CD', 'NNS', 'JJ', ',', 'MD']
Tag-tree (top rules)  :
                                          S                                                 
                  ________________________|_______________________________________________   
                 |                                        VP                              | 
                 |                     ___________________|___                            |  
                 NP                   |                       VP                          | 
      ___________|________________    |    ___________________|___________________        |  
     |       |          ADJP      |   |   |       |               PP              |       | 
     |       |        ___|____    |   |   |       |        _______|___            |       |  
     NP      |       NP       |   |   |   |       NP      |           NP          NP      | 
  ___|___    |    ___|___     |   |   |   |    ___|___    |  

### The statistical POS tagger

In [161]:
import pickle
from nltk import DefaultTagger, UnigramTagger, BigramTagger, TrigramTagger

class StatisticalPOSTagger:
    def __init__(self, default_tag='NN'):
        self.default_tag = default_tag
        self.tagger = None

    def train(self, tagged):
        t0 = DefaultTagger(self.default_tag)          # last resort: guess NN
        t1 = UnigramTagger(tagged, backoff=t0)        # word's most common tag
        t2 = BigramTagger(tagged, backoff=t1)         # + previous 1 tag
        t3 = TrigramTagger(tagged, backoff=t2)        # + previous 2 tags
        self.tagger = t3
        return self

    def tag(self, tokens):
        return self.tagger.tag(tokens)

    def accuracy(self, gold):
        return self.tagger.accuracy(gold)

    def save(self, path):
        with open(path, 'wb') as fh: pickle.dump(self.tagger, fh)

    @classmethod
    def load(cls, path):
        obj = cls()
        with open(path, 'rb') as fh: obj.tagger = pickle.load(fh)
        return obj

print('Tagger class defined.')

Tagger class defined.


In [162]:
import random

clean_tagged = [strip_traces(s) for s in tagged_sents]
clean_tagged = [s for s in clean_tagged if s]

random.seed(42)
idx = list(range(len(clean_tagged)))
random.shuffle(idx)
cut = int(0.9 * len(idx))
train_set = [clean_tagged[i] for i in idx[:cut]]
test_set  = [clean_tagged[i] for i in idx[cut:]]

held = StatisticalPOSTagger().train(train_set)
print('Held-out tagging accuracy: %.4f' % held.accuracy(test_set))

Held-out tagging accuracy: 0.9051


In [163]:
tagger = StatisticalPOSTagger().train(clean_tagged)
tagger.save('models/tagger.pkl')

# quick sanity check on a fresh sentence
demo = nltk.word_tokenize('The government approved a new economic plan yesterday.')
print(tagger.tag(demo))

[('The', 'DT'), ('government', 'NN'), ('approved', 'VBD'), ('a', 'DT'), ('new', 'JJ'), ('economic', 'JJ'), ('plan', 'NN'), ('yesterday', 'NN'), ('.', '.')]


### Inducing the PCFG

In [164]:
import re
from collections import Counter
from nltk import Nonterminal, induce_pcfg

def induce_grammar(parsed, horz_markov=2, vert_markov=0, prune=2):
    productions = []
    for t in parsed:
        tag_tree = to_tag_tree(t)
        if not isinstance(tag_tree, Tree):
            continue
        wrapped = Tree('TOP', [tag_tree])                 # single start symbol
        wrapped.collapse_unary(collapsePOS=False, collapseRoot=False)
        wrapped.chomsky_normal_form(horzMarkov=horz_markov, vertMarkov=vert_markov)
        productions.extend(wrapped.productions())
    if prune and prune > 1:                               # drop very rare rules
        counts = Counter(productions)
        productions = [p for p in productions
                       if counts[p] >= prune
                       or (len(p.rhs()) == 1 and isinstance(p.rhs()[0], str))]
    return induce_pcfg(Nonterminal('TOP'), productions)

grammar = induce_grammar(parsed_sents)
print('Grammar induced.')
print('  start symbol :', grammar.start())
print('  productions  :', len(grammar.productions()))
print('\nA few example rules:')
for p in grammar.productions()[:6]:
    print('  ', p)

Grammar induced.
  start symbol : TOP
  productions  : 2766

A few example rules:
   TOP -> S [0.905955]
   S -> NP S|<VP-.> [0.292545]
   NP -> NP NP|<,-ADJP> [0.000499085]
   NP -> NNP NNP [0.0412244]
   NNP -> 'NNP' [1.0]
   NP|<,-ADJP> -> , NP|<ADJP-,> [0.583333]


### Save the grammar (the "PCFG parser data" deliverable)

save it two ways:
- **`grammar.pcfg`** — a human-readable text file. *This is the PCFG deliverable.*
- **`grammar.pkl`** — a pickled copy the program reloads (CNF symbol names contain
  characters NLTK's text reader can't parse back, so reload from the pickle).

In [165]:
def clean_output_label(label):
    "Strip CNF/binarisation markers like |<...> from a node label."
    return re.sub(r'([|^]<[^>]*>)+', '', label)

def save_grammar(grammar, path):
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write('# Induced PCFG over Penn Treebank POS tags\n')
        fh.write('# start symbol: %s\n' % grammar.start())
        fh.write('# productions: %d\n\n' % len(grammar.productions()))
        for p in grammar.productions():
            fh.write('%s\n' % p)

save_grammar(grammar, 'models/grammar.pcfg')
with open('models/grammar.pkl', 'wb') as fh:
    pickle.dump(grammar, fh)
print('Saved models/grammar.pcfg and models/grammar.pkl')

Saved models/grammar.pcfg and models/grammar.pkl


### The CKY Viterbi parser



In [166]:
from collections import defaultdict

class CKYViterbiParser:
    def __init__(self, grammar):
        self.start = grammar.start()
        self.lex   = defaultdict(list)   # terminal   -> [(A, logp)]
        self.left  = defaultdict(list)   # left child -> [(C, A, logp)]
        self.unary = defaultdict(list)   # child      -> [(A, logp)]
        for p in grammar.productions():
            A, lp, rhs = p.lhs(), p.logprob(), p.rhs()
            if len(rhs) == 1 and isinstance(rhs[0], str):
                self.lex[rhs[0]].append((A, lp))
            elif len(rhs) == 1:
                self.unary[rhs[0]].append((A, lp))
            else:
                self.left[rhs[0]].append((rhs[1], A, lp))

    def _unary_closure(self, cell):
        agenda = list(cell.keys())
        while agenda:
            B = agenda.pop()
            bp = cell[B][0]
            for A, lp in self.unary.get(B, []):
                score = bp + lp
                if A not in cell or score > cell[A][0]:
                    cell[A] = (score, ('u', B))
                    agenda.append(A)

    def parse(self, tags):
        n = len(tags)
        T = [[None] * (n + 1) for _ in range(n + 1)]
        for i, tag in enumerate(tags):                       # width-1 cells
            cell = {A: (lp, ('t', tag)) for A, lp in self.lex.get(tag, [])}
            self._unary_closure(cell)
            T[i][i + 1] = cell
        for span in range(2, n + 1):                         # wider cells
            for i in range(0, n - span + 1):
                j = i + span
                cell = {}
                for k in range(i + 1, j):
                    L, R = T[i][k], T[k][j]
                    if not L or not R:
                        continue
                    for B in L:
                        lB = L[B][0]
                        for C, A, lp in self.left.get(B, ()):
                            if C in R:
                                score = lB + R[C][0] + lp
                                if A not in cell or score > cell[A][0]:
                                    cell[A] = (score, ('b', k, B, C))
                self._unary_closure(cell)
                T[i][j] = cell
        top = T[0][n]
        if self.start not in top:
            return None, None
        return self._build(T, 0, n, self.start), top[self.start][0]

    def _build(self, T, i, j, sym):
        kind, *rest = T[i][j][sym][1]
        if kind == 't':
            return Tree(str(sym), [rest[0]])
        if kind == 'u':
            return Tree(str(sym), [self._build(T, i, j, rest[0])])
        k, B, C = rest
        return Tree(str(sym), [self._build(T, i, k, B), self._build(T, k, j, C)])

    def parse_with_fallback(self, tags):
        "Return (tree, logprob, full_parse). Flat TOP tree if no full parse."
        tree, logp = self.parse(tags)
        if tree is not None:
            return tree, logp, True
        flat = Tree(clean_output_label(str(self.start)), [Tree(t, [t]) for t in tags])
        return flat, None, False

parser = CKYViterbiParser(grammar)
print('Parser ready.')

Parser ready.


In [167]:
# Demo: parse a tag sequence directly and show the (CNF) tree
tree, logp = parser.parse(['DT', 'NN', 'VBD', 'DT', 'JJ', 'NN', '.'])
print('log-probability: %.2f\n' % logp)
tree.pretty_print()

log-probability: -15.26

                TOP                                
                 |                                  
                 S                                 
      ___________|_________                         
     |                  S|<VP-.>                   
     |                _____|_____________________   
     |               VP                          | 
     |        _______|_____                      |  
     |       |             NP                    | 
     |       |    _________|_________            |  
     NP      |   |               NP|<JJ-NN>      | 
  ___|___    |   |          _________|_______    |  
 DT      NN VBD  DT        JJ                NN  . 
 |       |   |   |         |                 |   |  
 DT      NN VBD  DT        JJ                NN  . 



### End-to-end: words in, tree out

Now connect the two models. `POSPCFGParser.parse_tokens`:
1. tags the words,
2. parses the tag sequence (with the flat fallback so always get a tree),
3. puts the **real words** back on the leaves,
4. undoes the CNF binarisation and cleans helper labels for readability.

In [168]:
def _iter_leaf_positions(tree, path=()):
    for i, child in enumerate(tree):
        if isinstance(child, Tree):
            yield from _iter_leaf_positions(child, path + (i,))
        else:
            yield path + (i,)

def attach_words(tree, words):
    for pos_path, word in zip(list(_iter_leaf_positions(tree)), words):
        node = tree
        for idx in pos_path[:-1]:
            node = node[idx]
        node[pos_path[-1]] = word
    return tree

class POSPCFGParser:
    def __init__(self, tagger, parser):
        self.tagger, self.parser = tagger, parser
    def parse_tokens(self, tokens):
        tagged = self.tagger.tag(tokens)
        tags = [t for _, t in tagged]
        tree, logp, full = self.parser.parse_with_fallback(tags)
        tree = attach_words(tree, tokens)
        try:
            tree.un_chomsky_normal_form()
        except Exception:
            pass
        for st in tree.subtrees():
            st.set_label(clean_output_label(st.label()))
        return tagged, tree, logp, full

full_parser = POSPCFGParser(tagger, parser)

tagged, tree, logp, full = full_parser.parse_tokens(
    nltk.word_tokenize('Police arrested three suspects after a robbery in Colombo.'))
print('Full parse:', full, '  log-prob: %.2f\n' % (logp or 0))
tree.pretty_print()

Full parse: True   log-prob: -42.38

                                TOP                                         
                                 |                                           
                                 S                                          
   ______________________________|________________________________________   
  |               VP                                                      | 
  |        _______|______                                                 |  
  |       |             SBAR                                              | 
  |       |              |                                                |  
  |       |              S                                                | 
  |       |        ______|____________                                    |  
  |       |       |                   VP                                  | 
  |       |       |       ____________|_________                          |  
  |       |       |      |       

### 150 Daily Mirror sentences

In [169]:
import csv, os, io, unicodedata

CSV_PATH = 'data/daily_mirror_sentences.csv'      # columns: id, sentence, url

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        "Missing %s\nCreate it with a header row and columns: id, sentence, url." % CSV_PATH)

_SUBS = {'\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
         '\u2013': '-', '\u2014': '-', '\u2026': '...', '\u00a0': ' '}
def _clean(s):
    for a, b in _SUBS.items():
        s = s.replace(a, b)
    s = unicodedata.normalize('NFKD', s)                 
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return ' '.join(s.split()).strip()

def _read_text(path):
    """Excel on Windows saves CSV as cp1252, not UTF-8; try both."""
    raw = open(path, 'rb').read()
    for enc in ('utf-8-sig', 'cp1252', 'latin-1'):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw.decode('utf-8', errors='replace')

def load_sentence_csv(path):
    reader = csv.DictReader(io.StringIO(_read_text(path)))
    cols = {h.lower().strip(): h for h in reader.fieldnames}
    id_k, sen_k, url_k = cols.get('id'), cols.get('sentence'), cols.get('url')
    if sen_k is None:
        raise ValueError("CSV needs a 'sentence' column. Found: %s" % reader.fieldnames)
    rows, skipped = [], []
    for n, r in enumerate(reader, 1):
        sent = _clean(r.get(sen_k) or '')
        rid  = (r.get(id_k) or '').strip() if id_k else ''
        if not sent:
            skipped.append(rid or str(n)); continue
        url = _clean(r.get(url_k) or '') if url_k else ''
        rows.append({'id': rid or str(n), 'sentence': sent, 'url': url})
    if skipped:
        print('WARNING: skipped %d blank row(s): id(s) %s' % (len(skipped), ', '.join(skipped)))
    return rows

rows      = load_sentence_csv(CSV_PATH)
ids       = [r['id'] for r in rows]
raw_sents = [r['sentence'] for r in rows]
urls      = [r['url'] for r in rows]
print('Loaded', len(rows), 'sentences from', CSV_PATH)
if len(rows) != 150:
    print('NOTE: expected 150 sentences, found %d - check CSV.' % len(rows))
print('First row:', rows[0])

Loaded 150 sentences from data/daily_mirror_sentences.csv
First row: {'id': '1', 'sentence': 'The Canada Border Services Agency has seized suitcases packed with illicit substances bound for Sri Lanka at Toronto Pearson International Airport earlier this month.', 'url': 'https://www.dailymirror.lk/breaking-news/44-kg-of-hashish--13-kg-of-cocaine-destined-for-Sri-Lanka-seized-at-Toronto-Airport/108-351208'}


In [170]:
def tagged_to_str(tagged):
    """[(w, t), ...]  ->  'w/t w/t ...'"""
    return ' '.join('%s/%s' % (w, t) for w, t in tagged)

def str_to_tagged(s):
    """'w/t w/t ...'  ->  [(w, t), ...]   (splits on the LAST '/')"""
    out = []
    for tok in s.split():
        if '/' in tok:
            w, t = tok.rsplit('/', 1)
            out.append((w, t))
    return out

# tag every sentence; sentence_id points back to the id in the input CSV
auto_rows = []
for i, r in enumerate(rows, 1):
    tagged = tagger.tag(nltk.word_tokenize(r['sentence']))
    auto_rows.append({'id': i, 'sentence_id': r['id'],
                      'tagged': tagged_to_str(tagged)})

AUTO_CSV = 'data/daily_mirror_auto_tagged.csv'
with open(AUTO_CSV, 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=['id', 'sentence_id', 'tagged'])
    w.writeheader()
    w.writerows(auto_rows)
print('Wrote', AUTO_CSV, '(columns: id, sentence_id, tagged)')

print('\nExample  ->  id', auto_rows[0]['id'], '| sentence_id', auto_rows[0]['sentence_id'])
print('  tagged:', auto_rows[0]['tagged'])

Wrote data/daily_mirror_auto_tagged.csv (columns: id, sentence_id, tagged)

Example  ->  id 1 | sentence_id 1
  tagged: The/DT Canada/NNP Border/NN Services/NNPS Agency/NNP has/VBZ seized/VBD suitcases/NN packed/JJ with/IN illicit/NN substances/NN bound/NN for/IN Sri/NN Lanka/NN at/IN Toronto/NNP Pearson/NN International/NNP Airport/NN earlier/JJR this/DT month/NN ./.


### Manual annotation

In [171]:
# ---- Manual gold FIRST-PASS via an INDEPENDENT tagger ----
# nltk.pos_tag uses the averaged-perceptron tagger, which is separate from the
# statistical tagger I built. It gives an accurate starting point for the manual
from nltk import pos_tag

GOLD_CSV = 'data/daily_mirror_gold_tagged.csv'
if not os.path.exists(GOLD_CSV):
    gp = [{'id': r['id'], 'sentence_id': r['id'],
           'tagged': tagged_to_str(pos_tag(nltk.word_tokenize(r['sentence'])))}
          for r in rows]
    with open(GOLD_CSV, 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=['id', 'sentence_id', 'tagged'])
        w.writeheader(); w.writerows(gp)
    print('Created FIRST-PASS manual gold ->', GOLD_CSV)
    print('   VERIFY it using results/manual_annotation_review.csv (check the CHECK rows).')
else:
    print('Manual gold already exists ->', GOLD_CSV, '(kept, not overwritten).')

Manual gold already exists -> data/daily_mirror_gold_tagged.csv (kept, not overwritten).


In [172]:
# Hand-correction step:
#   1. copy  data/daily_mirror_auto_tagged.csv  ->  data/daily_mirror_gold_tagged.csv
#   2. open it and fix the wrong tags in the "tagged" column
#      e.g.  Colombo/NN -> Colombo/NNP,  arrested/VBN -> arrested/VBD
#   3. save and run this cell again.
GOLD_CSV = 'data/daily_mirror_gold_tagged.csv'

using = GOLD_CSV
if not os.path.exists(GOLD_CSV):
    using = AUTO_CSV

gold_rows = []
with open(using, encoding='utf-8-sig', newline='') as fh:
    for r in csv.DictReader(fh):
        gold_rows.append(r)

gold_ids   = [ (r.get('sentence_id') or r.get('id') or str(i))
               for i, r in enumerate(gold_rows, 1) ]
gold_sents = [ str_to_tagged(r['tagged']) for r in gold_rows ]
print('\nLoaded', len(gold_sents), 'gold sentences from', using)


Loaded 150 gold sentences from data/daily_mirror_gold_tagged.csv


### Two tagging methods: automatic vs manual gold

daily_mirror_gold_tagged.csv - Manual Tagging

In [173]:
# ---- Compare the TWO methods and AUTO-WRITE the review file ----
def _load_tagged_csv(path):
    with open(path, encoding='utf-8-sig', newline='') as fh:
        return {(r.get('sentence_id') or r.get('id')): str_to_tagged(r['tagged'])
                for r in csv.DictReader(fh)}

auto_map = _load_tagged_csv('data/daily_mirror_auto_tagged.csv')   
gold_map = _load_tagged_csv('data/daily_mirror_gold_tagged.csv')   

review_rows, tot, agree = [], 0, 0
for sid, gold in gold_map.items():
    for k, ((w, at), (_, mt)) in enumerate(zip(auto_map.get(sid, []), gold), 1):
        tot += 1; ok = (at == mt); agree += ok
        review_rows.append({'sentence_id': sid, 'word_index': k, 'word': w,
                            'auto_tag': at, 'manual_tag': mt,
                            'agree': '' if ok else 'CHECK'})

REVIEW_CSV = 'results/manual_annotation_review.csv'
with open(REVIEW_CSV, 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=['sentence_id', 'word_index', 'word',
                                       'auto_tag', 'manual_tag', 'agree'])
    w.writeheader(); w.writerows(review_rows)

print('Auto-wrote', REVIEW_CSV)
print('Tokens compared          :', tot)
print('Automatic == Manual gold : %.1f%% (%d tokens)' % (100*agree/max(tot,1), agree))
print('Rows to CHECK (disagree) : %d' % (tot-agree))
print('\nFirst 15 to check  (word: auto -> manual):')
for r in [x for x in review_rows if x['agree'] == 'CHECK'][:15]:
    print('   s%-3s %-16s %s -> %s' % (r['sentence_id'], r['word'], r['auto_tag'], r['manual_tag']))

Auto-wrote results/manual_annotation_review.csv
Tokens compared          : 3363
Automatic == Manual gold : 79.0% (2656 tokens)
Rows to CHECK (disagree) : 707

First 15 to check  (word: auto -> manual):
   s1   Border           NN -> NNP
   s1   Services         NNPS -> NNP
   s1   seized           VBD -> VBN
   s1   suitcases        NN -> NNS
   s1   packed           JJ -> VBD
   s1   illicit          NN -> JJ
   s1   substances       NN -> NNS
   s1   Sri              NN -> NNP
   s1   Lanka            NN -> NNP
   s1   Pearson          NN -> NNP
   s1   Airport          NN -> NNP
   s1   earlier          JJR -> RBR
   s2   CTV              NN -> NNP
   s2   CBSA             NN -> NNP
   s2   Sept             NN -> NNP


### Evaluation and report 

In [174]:
import time
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report)

known = {w.lower() for s in treebank.tagged_sents() for w, _ in s}

y_true, y_pred = [], []
known_tot = known_ok = oov_tot = oov_ok = 0
covered = 0
logps, times = [], []
parse_records = []      # (sentence_id, words, tree, full_parse)
sent_records  = []      # (sentence_id, n_tokens, tag_accuracy, full_parse)

for sid, sent in zip(gold_ids, gold_sents):
    words = [w for w, _ in sent]
    gtags = [t for _, t in sent]
    ptags = [t for _, t in tagger.tag(words)]
    s_ok = 0
    for w, g, p in zip(words, gtags, ptags):
        y_true.append(g); y_pred.append(p)
        hit = (g == p); s_ok += hit
        if w.lower() in known: known_tot += 1; known_ok += hit
        else:                  oov_tot   += 1; oov_ok   += hit
    t0 = time.time()
    tree, logp, full = parser.parse_with_fallback(ptags)
    times.append(time.time() - t0)
    if full:
        covered += 1; logps.append(logp)
    disp = attach_words(tree.copy(deep=True), words)
    try: disp.un_chomsky_normal_form()
    except Exception: pass
    for st in disp.subtrees(): st.set_label(clean_output_label(st.label()))
    parse_records.append((sid, words, disp, full))
    sent_records.append((sid, len(words), s_ok / max(len(words), 1), full))

print('Processed', len(gold_sents), 'sentences.')

Processed 150 sentences.


In [175]:
# ---- headline numbers ----
acc = accuracy_score(y_true, y_pred)
labels = sorted(set(y_true))
macro = precision_recall_fscore_support(y_true, y_pred, labels=labels,
                                        average='macro', zero_division=0)
weighted = precision_recall_fscore_support(y_true, y_pred, labels=labels,
                                           average='weighted', zero_division=0)

print('Token accuracy      : %.4f' % acc)
print('Macro    F1         : %.4f' % macro[2])
print('Weighted F1         : %.4f' % weighted[2])
print('Known-word accuracy : %.4f' % (known_ok / max(known_tot, 1)))
print('OOV-word accuracy   : %.4f' % (oov_ok / max(oov_tot, 1)))
print('Parse coverage      : %.4f (%d/%d full parses)'
      % (covered / len(gold_sents), covered, len(gold_sents)))
print('Avg parse time      : %.4f s' % (sum(times) / len(times)))

Token accuracy      : 0.7949
Macro    F1         : 0.7367
Weighted F1         : 0.7947
Known-word accuracy : 0.9201
OOV-word accuracy   : 0.2027
Parse coverage      : 0.9733 (146/150 full parses)
Avg parse time      : 0.8176 s


In [176]:
# ---- per-tag table ----
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

              precision    recall  f1-score   support

           $       1.00      1.00      1.00         3
          ''       0.89      0.89      0.89         9
           (       0.00      0.00      0.00         7
           )       0.00      0.00      0.00         7
           ,       1.00      1.00      1.00       126
           .       1.00      1.00      1.00       151
           :       1.00      1.00      1.00         9
          CC       1.00      1.00      1.00        95
          CD       1.00      0.64      0.78        58
          DT       1.00      1.00      1.00       306
          EX       1.00      1.00      1.00         2
          FW       0.00      0.00      0.00         1
          IN       0.98      0.97      0.97       346
          JJ       0.94      0.62      0.74       245
         JJR       0.67      0.77      0.71        13
         JJS       0.86      0.75      0.80         8
          MD       1.00      1.00      1.00        41
          NN       0.45    

In [177]:
# ---- most frequent tagging mistakes (gold -> predicted) ----
from collections import Counter
conf = Counter((g, p) for g, p in zip(y_true, y_pred) if g != p)
print('Top confusion pairs (gold -> predicted : count):')
for (g, p), c in conf.most_common(10):
    print('  %-5s -> %-5s : %d' % (g, p, c))

Top confusion pairs (gold -> predicted : count):
  NNP   -> NN    : 266
  JJ    -> NN    : 80
  NNS   -> NN    : 75
  VBG   -> NN    : 31
  VB    -> NN    : 24
  CD    -> NN    : 21
  VBN   -> VBD   : 18
  VBN   -> NN    : 18
  VBD   -> VBN   : 13
  VBZ   -> NN    : 10


In [178]:
import csv

with open('results/evaluation_report.md', 'w', encoding='utf-8') as fh:
    fh.write('# Evaluation Report - Statistical POS-tagged PCFG Parser\n\n')
    fh.write('Test set: %d Daily Mirror sentences, %d tokens.\n\n'
             % (len(gold_sents), len(y_true)))
    fh.write('## Dataset\n\nTrained on the labelled NLTK Penn Treebank '
             '(`nltk.corpus.treebank`), as the brief instructs. The Kaggle '
             'language-modelling file has no tags or trees and cannot be used.\n\n')
    fh.write('## Tagging metrics\n\n')
    fh.write('| Metric | Value |\n|---|---|\n')
    fh.write('| Token accuracy | %.4f |\n' % acc)
    fh.write('| Macro F1 | %.4f |\n' % macro[2])
    fh.write('| Weighted F1 | %.4f |\n' % weighted[2])
    fh.write('| Known-word accuracy | %.4f |\n' % (known_ok / max(known_tot, 1)))
    fh.write('| OOV-word accuracy | %.4f |\n' % (oov_ok / max(oov_tot, 1)))
    fh.write('\n## Parsing metrics\n\n')
    fh.write('| Metric | Value |\n|---|---|\n')
    fh.write('| Full-parse coverage | %.4f (%d/%d) |\n'
             % (covered / len(gold_sents), covered, len(gold_sents)))
    fh.write('| Avg parse time (s) | %.4f |\n' % (sum(times) / len(times)))
    fh.write('\n## Per-tag detail\n\n```\n')
    fh.write(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    fh.write('```\n\n## Note on PARSEVAL\n\nFull bracketing (PARSEVAL) scores '
             'need hand-built gold parse *trees* for all 150 sentences, which is '
             'out of scope here; token-level tagging metrics plus parse coverage '
             'are the standard metrics reported.\n')

# per-tag precision / recall / F1
with open('results/tagging_metrics.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['tag', 'precision', 'recall', 'f1', 'support'])
    P, R, F, S = precision_recall_fscore_support(y_true, y_pred, labels=labels,
                                                 zero_division=0)
    for i, tag in enumerate(labels):
        w.writerow([tag, '%.4f' % P[i], '%.4f' % R[i], '%.4f' % F[i], S[i]])

# per-sentence results, traceable by sentence_id
with open('results/per_sentence.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh)
    w.writerow(['sentence_id', 'n_tokens', 'tag_accuracy', 'full_parse'])
    for sid, n, a, full in sent_records:
        w.writerow([sid, n, '%.4f' % a, full])

# the parse tree for every sentence, labelled with its sentence_id
with open('results/parses.txt', 'w', encoding='utf-8') as fh:
    for sid, words, tree, full in parse_records:
        fh.write('### sentence_id %s: %s\n' % (sid, ' '.join(words)))
        if not full:
            fh.write('(flat fallback - not a full parse)\n')
        fh.write(tree.pformat() + '\n\n')

print('Wrote results/evaluation_report.md, tagging_metrics.csv, '
      'per_sentence.csv, parses.txt')

Wrote results/evaluation_report.md, tagging_metrics.csv, per_sentence.csv, parses.txt


### Deeper analysis

In [179]:
# ---- Analysis 1: parse coverage by sentence length ----
import collections, csv
buckets = collections.OrderedDict([('08-15', []), ('16-25', []), ('26-35', []), ('36+', [])])
for sid, n, a, full in sent_records:
    key = '08-15' if n <= 15 else '16-25' if n <= 25 else '26-35' if n <= 35 else '36+'
    buckets[key].append(1 if full else 0)

with open('results/coverage_by_length.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['length_band', 'sentences', 'full_parses', 'coverage'])
    print('length band | sentences | full parses | coverage')
    for band, vals in buckets.items():
        n = len(vals); c = sum(vals); cov = c / n if n else 0
        w.writerow([band, n, c, '%.3f' % cov])
        print('  %-7s   |   %4d    |    %4d     |  %5.1f%%' % (band, n, c, 100 * cov))
print('\nWrote results/coverage_by_length.csv')

length band | sentences | full parses | coverage
  08-15     |     40    |      40     |  100.0%
  16-25     |     53    |      52     |   98.1%
  26-35     |     49    |      46     |   93.9%
  36+       |      8    |       8     |  100.0%

Wrote results/coverage_by_length.csv


In [180]:
# ---- Analysis 2: out-of-vocabulary (unseen) words and how they were tagged ----
import collections, csv
oov_count = collections.Counter()
oov_seen = {}
for sent in gold_sents:
    words = [w for w, _ in sent]
    pred = [t for _, t in tagger.tag(words)]
    for (w, g), p in zip(sent, pred):
        if w.lower() not in known:
            oov_count[w] += 1
            oov_seen.setdefault(w, (g, p))              # (gold, predicted)

rows = [(w, c, oov_seen[w][0], oov_seen[w][1],
         'ok' if oov_seen[w][0] == oov_seen[w][1] else 'wrong')
        for w, c in oov_count.most_common()]
with open('results/oov_words.csv', 'w', newline='', encoding='utf-8') as fh:
    wr = csv.writer(fh); wr.writerow(['word', 'count', 'gold_tag', 'predicted_tag', 'result'])
    wr.writerows(rows)

print('Distinct OOV words:', len(rows))
print('Most frequent OOV words  (word: gold vs predicted):')
for w, c, g, p, res in rows[:15]:
    print('  %-16s x%-2d  gold=%-4s pred=%-4s  %s' % (w, c, g, p, res))
print('\nWrote results/oov_words.csv')

Distinct OOV words: 396
Most frequent OOV words  (word: gold vs predicted):
  Sri              x13  gold=NNP  pred=NN    wrong
  Lanka            x11  gold=NNP  pred=NN    wrong
  Iran             x7   gold=NNP  pred=NN    wrong
  (                x7   gold=(    pred=NN    wrong
  )                x7   gold=)    pred=NN    wrong
  solar            x6   gold=JJ   pred=NN    wrong
  Iranian          x6   gold=JJ   pred=NN    wrong
  Yanbu            x6   gold=NNP  pred=NN    wrong
  GS1              x6   gold=NNP  pred=NN    wrong
  Colombo          x5   gold=NNP  pred=NN    wrong
  Hormuz           x5   gold=NNP  pred=NN    wrong
  Houthis          x5   gold=NNP  pred=NN    wrong
  infrastructure   x5   gold=NN   pred=NN    ok
  ICAO             x5   gold=NNP  pred=NN    wrong
  digital          x5   gold=JJ   pred=NN    wrong

Wrote results/oov_words.csv


In [181]:
# ---- Analysis 3: full tagging confusion list (gold -> predicted) ----
import collections, csv
conf = collections.Counter((g, p) for g, p in zip(y_true, y_pred) if g != p)
with open('results/confusions.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['gold_tag', 'predicted_tag', 'count'])
    for (g, p), c in conf.most_common():
        w.writerow([g, p, c])
print('Distinct confusion types:', len(conf), '| total mistakes:', sum(conf.values()))
print('Top 10 confusions  (gold -> predicted : count):')
for (g, p), c in conf.most_common(10):
    print('  %-5s -> %-5s : %d' % (g, p, c))
print('\nWrote results/confusions.csv')

Distinct confusion types: 59 | total mistakes: 690
Top 10 confusions  (gold -> predicted : count):
  NNP   -> NN    : 266
  JJ    -> NN    : 80
  NNS   -> NN    : 75
  VBG   -> NN    : 31
  VB    -> NN    : 24
  CD    -> NN    : 21
  VBN   -> VBD   : 18
  VBN   -> NN    : 18
  VBD   -> VBN   : 13
  VBZ   -> NN    : 10

Wrote results/confusions.csv
